# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# Write your solution here

result_df.write.mode("overwrite").parquet("/FileStore/tables/output/fachoursbymonth.parquet")

# verify
spark.read.parquet("/FileStore/tables/output/fachoursbymonth.parquet").show()

## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here
result_df.write.partitionBy("facid").mode("overwrite").saveAsTable("threejoin_delta")


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Write your solution here
import requests
from pyspark.sql import Row
from pyspark.sql.functions import to_date, weekofyear, year, col, max as spark_max

API_KEY = dbutils.secrets.get(scope="your-scope", key="rapidapi-key")  # don't hardcode this
symbols = ["GOOGL", "AAPL", "MSFT", "TSLA"]

url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {"X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com", "X-RapidAPI-Key": API_KEY}

all_dfs = []
for symbol in symbols:
    querystring = {"function": "TIME_SERIES_DAILY", "symbol": symbol, "outputsize": "compact", "datatype": "json"}
    resp = requests.get(url, headers=headers, params=querystring)
    data = resp.json().get("Time Series (Daily)", {})

    rows = [Row(symbol=symbol, date=d, close=float(v["4. close"])) for d, v in data.items()]
    all_dfs.append(spark.createDataFrame(rows))

stocks_df = all_dfs[0]
for d in all_dfs[1:]:
    stocks_df = stocks_df.union(d)

stocks_df = stocks_df.withColumn("date", to_date("date"))
stocks_df = stocks_df.withColumn("week", weekofyear("date")).withColumn("year", year("date"))

weekly_max = (stocks_df.groupBy("symbol", "year", "week")
    .agg(spark_max("close").alias("max_closing_price")))

weekly_max.write.partitionBy("symbol").mode("overwrite").saveAsTable("max_closing_price_weekly")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-8148423317562115>, line 6
      3 from pyspark.sql import Row
      4 from pyspark.sql.functions import to_date, weekofyear, year, col, max as spark_max
----> 6 API_KEY = dbutils.secrets.get(scope="your-scope", key="rapidapi-key")  # don't hardcode this
      7 symbols = ["GOOGL", "AAPL", "MSFT", "TSLA"]
      9 url = "https://alpha-vantage.p.rapidapi.com/query"

File /databricks/python_shell/lib/dbruntime/dbutils.py:422, in DBUtils.SecretsHandler.get(self, scope, catalog, schema, key, *posArgs)
    420     scope = argsToPass[0]
    421     key = argsToPass[1]
--> 422     return self._credentials_routing_client.get_secret_by_scope(scope, key)
    423 else:
    424     catalog = argsToPass[0]

File /databricks/python_shell/lib/dbruntime/driver_connection/routing_client/credentials_routing_client.py:34, in CredentialsRouting

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Write your solution here

jdbc_url = "jdbc:postgresql://<host>:<port>/<database>"

rna_df = (spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT * FROM rna LIMIT 100")
    .option("user", "<username from their docs>")
    .option("driver", "org.postgresql.Driver")
    .load())

rna_df.write.mode("overwrite").saveAsTable("rna_100_records")